# 12 — Xuất head OS (05b) sang TensorFlow Lite

Đầu vào: một run `Models/<ts>_os/model.joblib` của `05b_train_os_head`. Đầu ra nằm cạnh nó, trong `Models/<ts>_os/tflite/`:

| File | Nội dung |
|---|---|
| `sdc_os_forest.tflite` | rừng L2: `float32[N, 187]` (vector đã encode) → `float32[N, 5]` xác suất, **kèm contract nhúng trong metadata `sdc.contract`** |
| `sdc_os_contract.json` | bản sao contract để người đọc — byte-for-byte giống bản nhúng (sha256 ghi trong `meta.json`) |
| `meta.json` | run nguồn, kích thước, sha256 contract, kết quả parity |

Contract là mọi thứ phía thiết bị phải tự làm: encode chuỗi, L0, L1, ngưỡng, luật chặn. Nhúng nó vào chính
file `.tflite` (như `metadata_props["sdc.contract"]` của ONNX ở notebook 09) thì chỉ deploy một file và model
không thể lệch vocab với contract. Đọc ra trên thiết bị: C++ `FlatBufferModel::ReadAllMetadata()`, hoặc duyệt
flatbuffer `Model.metadata` tìm tên `sdc.contract` rồi lấy `buffers[metadata.buffer].data`.

**Vì sao tách làm hai.** TFLite builtin không có op chuỗi: TF-IDF và `OrdinalEncoder` chỉ chạy được qua
Flex delegate (`SELECT_TF_OPS`), mà Flex kéo theo gần như cả runtime TensorFlow — không lên được router
OpenWrt (xem chú thích locale/musl ở notebook 11). Nên graph nhận vector số; encode làm ở phía thiết bị
theo đúng vocab ghi trong contract. Mục 5 chứng minh encoder viết lại từ contract cho ra **cùng một vector**
với `ColumnTransformer`.

**Vì sao không đi đường sklearn → ONNX → TF.** skl2onnx sinh `TreeEnsembleClassifier` (domain `ai.onnx.ml`);
onnx2tf/onnx-tf không chuyển được op này, còn TF Decision Forests thì không xuất ra TFLite. Rừng ở đây nhỏ
(300 cây, ~13k node, sâu ≤ 20) nên dựng thẳng bằng `GATHER`/`LESS_EQUAL`/`SELECT_V2`: mỗi vòng lặp đưa
mọi cây xuống một tầng, lặp đúng `max_depth` lần — lá trỏ về chính nó nên đứng yên.

**Cần TensorFlow** trong kernel (Windows + Python 3.14 hiện chỉ có bản `--pre`: `pip install --pre tensorflow`),
nên thêm `ai-edge-litert` để parity chạy bằng đúng runtime của thiết bị. Mục 4 đọc `sessions_verified.parquet`
nếu có `pyarrow`, không có thì chỉ đo trên CSV.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
# Nạp định nghĩa dùng chung từ 00_common.ipynb (không train).
from pathlib import Path

_cwd = Path.cwd().resolve()
_PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                  for d in (p, p / "pipeline", p / "Code" / "pipeline")
                  if (d / "00_common.ipynb").is_file()), None)
assert _PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_PIPELINE / "00_common.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import ast
import json
import re
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from tensorflow.lite.python import schema_py_generated as tfl_schema
from tensorflow.lite.tools import flatbuffer_utils

# Kiểm bằng LiteRT — runtime mà thiết bị chạy — nếu có; `tf.lite.Interpreter` đã deprecated từ TF 2.20.
try:
    from ai_edge_litert.interpreter import Interpreter
    RUNTIME = "ai_edge_litert"
except ImportError:
    Interpreter = tf.lite.Interpreter
    RUNTIME = "tf.lite"

print("TensorFlow", tf.__version__, "| interpreter:", RUNTIME)

Configured for 88-core CPU
Gốc dự án: D:\01.AI_Security\02.SDC


Đã nạp hàm SDC từ 00_common.ipynb


TensorFlow 2.22.0-rc0 | interpreter: ai_edge_litert


## 1. Chọn run OS

`RUN = None` lấy run `*_os` mới nhất. Luật L0/L1 và hàm quyết định **không nằm trong** `model.joblib` mà là
code trong `05b`, nên nạp thẳng định nghĩa từ notebook đó (theo tên, qua AST) thay vì chép lại — contract
và parity ở dưới luôn bám đúng bản 05b hiện hành.

In [3]:
RUN = "20261002_163153_os"     # None = run *_os mới nhất

run_dir = (MODELS / RUN) if RUN else sorted(p for p in MODELS.glob("*_os")
                                            if (p / "model.joblib").is_file())[-1]
bundle = joblib.load(run_dir / "model.joblib")
assert bundle["format"] == "sdc-os-v0", bundle["format"]
forest, enc = bundle["model"], bundle["encoder"]
CLASSES = [str(c) for c in forest.classes_]
N_FEATURES = int(forest.n_features_in_)
print(f"run={run_dir.name}  cây={len(forest.estimators_)}  feature={N_FEATURES}  lớp={CLASSES}")
missing_cls = sorted(set(bundle["classes"]) - set(CLASSES))
if missing_cls:
    print("Lớp khai báo nhưng rừng chưa từng thấy mẫu (không bao giờ được rừng trả):", missing_cls)

# Định nghĩa quyết định OS lấy từ 05b — một nguồn duy nhất.
NAMES_05B = ["UNK", "OS_THRESHOLD", "NOT_PRESENT", "HOST_OS", "mac_is_random", "prl_set",
             "l0_os", "l1_os", "os_decide"]
_nb05b = json.loads((_PIPELINE / "train_model" / "05b_train_os_head.ipynb").read_text(encoding="utf-8"))
_wanted = []
for _cell in _nb05b["cells"]:
    if _cell["cell_type"] != "code":
        continue
    try:
        _tree = ast.parse("".join(_cell["source"]))
    except SyntaxError:         # cell có magic IPython — không chứa định nghĩa cần lấy
        continue
    for _node in _tree.body:
        _name = (_node.name if isinstance(_node, (ast.FunctionDef, ast.ClassDef)) else
                 _node.targets[0].id if isinstance(_node, ast.Assign) and isinstance(_node.targets[0], ast.Name)
                 else None)
        if _name in NAMES_05B:
            _wanted.append(_node)
OS05B = {"re": re, "np": np, "MISSING": MISSING, "apply_encoder": apply_encoder}
exec(compile(ast.Module(body=_wanted, type_ignores=[]), "05b_train_os_head", "exec"), OS05B)
assert set(NAMES_05B) <= set(OS05B), sorted(set(NAMES_05B) - set(OS05B))
assert OS05B["OS_THRESHOLD"] == bundle["threshold"], "ngưỡng trong 05b lệch với run đã lưu"
print("Đã nạp từ 05b:", ", ".join(NAMES_05B))

run=20261002_163153_os  cây=300  feature=187  lớp=['Android', 'Embedded', 'Linux', 'Windows', 'iOS']
Lớp khai báo nhưng rừng chưa từng thấy mẫu (không bao giờ được rừng trả): ['macOS']
Đã nạp từ 05b: UNK, OS_THRESHOLD, NOT_PRESENT, HOST_OS, mac_is_random, prl_set, l0_os, l1_os, os_decide


## 2. Rừng cây → mảng phẳng

Mọi node của 300 cây nối thành một mảng; `roots[t]` là chỉ số gốc cây `t`. Lá: `left = right = chính nó`,
nên duyệt thừa vòng vẫn đứng yên. Giá trị lá chuẩn hoá về phân phối lớp — `predict_proba` của rừng là
trung bình các phân phối này.

**Ngưỡng float32.** sklearn so `x_float32 <= threshold_float64`. Ép ngưỡng về float32 theo kiểu làm tròn
gần nhất có thể *nhảy lên* đúng bằng một giá trị x, lật phép so sánh. Lấy float32 lớn nhất **không vượt**
ngưỡng gốc thì với mọi x float32: `x <= t32 ⇔ x <= t64` — so sánh khớp tuyệt đối, không phải gần đúng.

In [4]:
def float32_floor(values):
    """float32 lớn nhất <= values (float64) — giữ nguyên kết quả của phép so x_f32 <= t."""
    out = values.astype(np.float32)
    up = out.astype(np.float64) > values
    out[up] = np.nextafter(out[up], np.float32(-np.inf))
    return out


def flatten_forest(model):
    feature, threshold, left, right, value, roots = [], [], [], [], [], []
    offset = 0
    for est in model.estimators_:
        t = est.tree_
        n = t.node_count
        own = np.arange(n)
        leaf = t.children_left == -1
        # cây con nhận nhãn đã mã hoá 0..C-1 — cột value theo đúng thứ tự `model.classes_`
        assert t.value.shape[1:] == (1, len(model.classes_)), t.value.shape
        feature.append(np.where(leaf, 0, t.feature))
        threshold.append(np.where(leaf, 0.0, float32_floor(t.threshold)))
        left.append(np.where(leaf, own, t.children_left) + offset)
        right.append(np.where(leaf, own, t.children_right) + offset)
        v = t.value[:, 0, :]
        value.append(v / v.sum(axis=1, keepdims=True))
        roots.append(offset)
        offset += n
    return {
        "feature": np.concatenate(feature).astype(np.int32),
        "threshold": np.concatenate(threshold).astype(np.float32),
        "left": np.concatenate(left).astype(np.int32),
        "right": np.concatenate(right).astype(np.int32),
        "value": np.concatenate(value).astype(np.float32),
        "roots": np.asarray(roots, dtype=np.int32),
        "depth": max(est.tree_.max_depth for est in model.estimators_),
    }


flat = flatten_forest(forest)
print(f"node={len(flat['feature'])}  cây={len(flat['roots'])}  sâu={flat['depth']}  "
      f"value={flat['value'].shape}")

node=13052  cây=300  sâu=20  value=(13052, 5)


## 3. Graph TF chỉ dùng op builtin

Input `input` `[N, 187]`, output `output` `[N, 5]`, batch động. Gather trên `x` làm phẳng (`row * F + feature`) thay cho `batch_dims`, để
không phụ thuộc phiên bản kernel GATHER của runtime trên thiết bị. Sau khi convert, liệt kê op trong file và
**fail** nếu có op ngoài builtin.

In [5]:
IO_NAMES = ("input", "output")   # tên tensor + khoá signature — cùng quy ước với ONNX ở notebook 09


class ForestModule(tf.Module):
    def __init__(self, flat, n_features):
        super().__init__()
        self.n_features = n_features
        self.depth = int(flat["depth"])
        self.feature = tf.constant(flat["feature"])
        self.threshold = tf.constant(flat["threshold"])
        self.left = tf.constant(flat["left"])
        self.right = tf.constant(flat["right"])
        self.value = tf.constant(flat["value"])
        self.roots = tf.constant(flat["roots"])
        self.predict = tf.function(self._predict, input_signature=[
            tf.TensorSpec([None, n_features], tf.float32, name=IO_NAMES[0])])

    def _predict(self, input):
        n = tf.shape(input)[0]
        x = tf.reshape(input, [-1])
        row = tf.expand_dims(tf.range(n) * self.n_features, 1)          # [N, 1]
        node = tf.tile(tf.expand_dims(self.roots, 0), tf.stack([n, 1]))  # [N, T]
        for _ in range(self.depth):
            xv = tf.gather(x, tf.gather(self.feature, node) + row)
            node = tf.where(xv <= tf.gather(self.threshold, node),
                            tf.gather(self.left, node), tf.gather(self.right, node))
        proba = tf.reduce_mean(tf.gather(self.value, node), axis=1)     # [N, C]
        return {IO_NAMES[1]: proba}


def rename_io(model_bytes, names=IO_NAMES):
    """Đặt tên tensor vào/ra và khoá signature thành `input`/`output`.

    Converter tự đặt tên tensor (`serving_default_input:0`, ra thì lấy tên op cuối, vd. `Shape`), không
    chỉnh được qua API. Sửa thẳng flatbuffer: chỉ đổi chuỗi tên, graph không đổi.
    """
    model = flatbuffer_utils.read_model_from_bytearray(bytearray(model_bytes))
    assert len(model.subgraphs) == 1
    sg = model.subgraphs[0]
    assert len(sg.inputs) == 1 and len(sg.outputs) == 1, (sg.inputs, sg.outputs)
    sg.tensors[sg.inputs[0]].name = names[0]
    sg.tensors[sg.outputs[0]].name = names[1]
    for sig in model.signatureDefs or []:
        sig.inputs[0].name, sig.outputs[0].name = names
    return bytes(flatbuffer_utils.convert_object_to_bytearray(model))


module = ForestModule(flat, N_FEATURES)
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [module.predict.get_concrete_function()], module)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_bytes = rename_io(converter.convert())

interp = Interpreter(model_content=tflite_bytes)
io = (interp.get_input_details()[0]["name"], interp.get_output_details()[0]["name"])
assert io == IO_NAMES, io
assert interp.get_signature_list() == {"serving_default": {"inputs": [IO_NAMES[0]], "outputs": [IO_NAMES[1]]}}
ops = sorted({op["op_name"] for op in interp._get_ops_details()})
print(f"TFLite: {len(tflite_bytes) / 1024:.0f} KiB, vào/ra: {io}, op: {ops}")
assert not [op for op in ops if op.startswith(("Flex", "Custom"))], ops

INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpti_1kozv\assets


INFO:tensorflow:Assets written to: C:\Users\admin\AppData\Local\Temp\tmpti_1kozv\assets


TFLite: 485 KiB, vào/ra: ('input', 'output'), op: ['ADD', 'EXPAND_DIMS', 'GATHER', 'LESS_EQUAL', 'MEAN', 'MUL', 'PACK', 'RANGE', 'RESHAPE', 'SELECT_V2', 'SHAPE', 'STRIDED_SLICE', 'TILE']


## 4. Parity: TFLite so với sklearn

Ba nguồn dữ liệu, encode bằng **chính encoder của run**:

* `sessions_verified.parquet` — đúng phân phối train (bỏ qua nếu máy không đọc được parquet);
* pcap tự thu (`session_features_capture.csv`) và `test_pcap_collection` — thiết bị/MAC mới;
* 20 000 vector ngẫu nhiên trong không gian đã encode — phủ những nhánh dữ liệu thật không đi tới.

Đo trên xác suất, trên nhãn argmax, và trên **quyết định cuối** (`os_decide` của 05b, thay rừng sklearn bằng
xác suất TFLite) vì quyết định có ngưỡng 0.6: chênh float32 ~1e-7 chỉ đáng kể nếu nó lật được ngưỡng.

In [6]:
def prep_os_frame(frame):
    """Đủ cột cho head OS, kiểu đúng như lúc train — như `prep()` của 05b."""
    f = frame.copy()
    f["mac_is_random"] = f["mac"].map(OS05B["mac_is_random"]) if "mac" in f else 0
    if "dhcp_hostname" not in f:
        f["dhcp_hostname"] = MISSING
    f["dhcp_hostname"] = f["dhcp_hostname"].fillna(MISSING).astype(str)
    for c in bundle["cols"]:
        numeric = c.startswith(("dhcp_opt_", "has_")) or c in ("dhcp_prl_len", "mac_is_random")
        if c not in f:
            f[c] = 0 if numeric else MISSING
        f[c] = f[c].fillna(0).astype(int) if numeric else f[c].fillna(MISSING).astype(str)
    return f.reset_index(drop=True)


frames = {}
try:
    frames["sessions_verified"] = prep_os_frame(load_sessions(SESSIONS_PATH)[0])
except Exception as exc:  # parquet engine bị chặn/thiếu trên một số máy
    print(f"Bỏ qua sessions_verified: {type(exc).__name__}: {str(exc).splitlines()[0][:120]}")

feat = DATA / "features"
cap = pd.read_csv(feat / "session_features_capture.csv")
dhcp_cap = pd.read_csv(feat / "dhcp_features_capture.csv")
cap["dhcp_hostname"] = cap["mac"].map(dhcp_cap.dropna(subset=["hostname"]).drop_duplicates("mac")
                                      .set_index("mac")["hostname"])
frames["capture"] = prep_os_frame(cap)
test3 = pd.read_csv(feat / "test_pcap_collection" / "session_features.csv")
test3["dhcp_hostname"] = test3["device"]
frames["test_pcap_collection"] = prep_os_frame(test3)


def random_encoded(n, seed=0):
    """Vector ngẫu nhiên đúng miền giá trị của từng khối encoder."""
    rng = np.random.default_rng(seed)
    X = np.zeros((n, N_FEATURES), dtype=np.float32)
    j = 0
    for c in enc["num_cols"]:
        X[:, j] = rng.integers(0, 25, n) if c == "dhcp_prl_len" else rng.integers(0, 2, n)
        j += 1
    for cats in enc["ct"].named_transformers_["cat"].categories_:
        X[:, j] = rng.integers(-1, len(cats), n)
        j += 1
    for c in enc["text_cols"]:
        k = len(enc["ct"].named_transformers_[f"t_{c}"].vocabulary_)
        block = rng.random((n, k)) * (rng.random((n, k)) < 3 / k)
        norm = np.linalg.norm(block, axis=1, keepdims=True)
        X[:, j:j + k] = np.divide(block, norm, out=np.zeros_like(block), where=norm > 0)
        j += k
    assert j == N_FEATURES
    return X


def tflite_proba(X, model_bytes=None):
    it = Interpreter(model_content=model_bytes or tflite_bytes)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    it.resize_tensor_input(inp["index"], [len(X), N_FEATURES])
    it.allocate_tensors()
    it.set_tensor(inp["index"], np.ascontiguousarray(X, dtype=np.float32))
    it.invoke()
    return it.get_tensor(out["index"])


class _TFLiteForest:
    """Thay `bundle['model']` khi gọi `os_decide` của 05b: cùng API, xác suất từ TFLite."""
    classes_ = forest.classes_

    def predict_proba(self, X):
        return tflite_proba(X).astype(np.float64)


tfl_bundle = {**bundle, "model": _TFLiteForest()}
rows = []
for name, frame in {**frames, "random": None}.items():
    X = random_encoded(20_000) if frame is None else apply_encoder(frame[bundle["cols"]], enc)[0]
    p_sk, p_tf = forest.predict_proba(X), tflite_proba(X)
    row = {"nguồn": name, "dòng": len(X), "max |Δp|": float(np.abs(p_sk - p_tf).max()),
           "argmax khác": int((p_sk.argmax(1) != p_tf.argmax(1)).sum())}
    if frame is not None:
        d_sk = OS05B["os_decide"](bundle, frame)
        d_tf = OS05B["os_decide"](tfl_bundle, frame)
        row["quyết định khác"] = sum(a[:2] != b[:2] for a, b in zip(d_sk, d_tf))
    rows.append(row)
parity = pd.DataFrame(rows).set_index("nguồn")
display(parity)
assert parity["argmax khác"].sum() == 0 and parity["quyết định khác"].fillna(0).sum() == 0
assert parity["max |Δp|"].max() < 1e-5

,dòng,max |Δp|,argmax khác,quyết định khác
nguồn,,,,
sessions_verified,8189,1.818320e-07,0,0.0
capture,62,1.818320e-07,0,0.0
test_pcap_collection,3,3.814697e-08,0,0.0
random,20000,6.021568e-08,0,NaN


## 5. Contract cho phía thiết bị

Mô tả đủ để code C/C++/Rust trên router dựng lại vector 187 chiều và ra cùng quyết định với `os_decide`:

* `features[i]` — cột thứ `i` của input, theo đúng thứ tự `ColumnTransformer` (num → cat → TF-IDF từng cột);
* `categorical` — chuỗi → chỉ số; không có trong danh sách → `-1`;
* `tfidf` — lowercase, tách token bằng `token_pattern` (`findall`), đếm token có trong vocab, nhân `idf`,
  chuẩn hoá L2 **trong từng khối cột**;
* `decision` — L0 (hostname, VCI) → L1 (Jaccard tập option DHCP, mDNS Apple) → rừng + ngưỡng + luật chặn
  cửa sổ chỉ-TLS.

Kiểm chứng: một encoder viết lại **chỉ từ JSON** (không dùng sklearn) phải ra đúng vector của
`ColumnTransformer` trên mọi dòng thật.

In [7]:
ct = enc["ct"]
features = [{"kind": "num", "col": c} for c in enc["num_cols"]]
features += [{"kind": "cat", "col": c} for c in enc["cat_cols"]]
categorical = {c: [str(v) for v in cats]
               for c, cats in zip(enc["cat_cols"], ct.named_transformers_["cat"].categories_)}
tfidf = {}
for c in enc["text_cols"]:
    vec = ct.named_transformers_[f"t_{c}"]
    vocab = [str(t) for t in vec.get_feature_names_out()]
    tfidf[c] = {"offset": len(features), "token_pattern": vec.token_pattern,
                "lowercase": bool(vec.lowercase), "norm": vec.norm,
                "sublinear_tf": bool(vec.sublinear_tf),
                "vocabulary": vocab, "idf": [float(v) for v in vec.idf_]}
    features += [{"kind": "tfidf", "col": c, "token": t} for t in vocab]
assert len(features) == N_FEATURES

signatures = [{"options": sorted(int(o) for o in key), "os": sorted(owners)}
              for key, owners in bundle["signatures"].items()]

contract = {
    "format": "sdc-os-tflite-v1",
    "source_run": run_dir.name,
    "created": datetime.now().isoformat(timespec="seconds"),
    "missing": MISSING,
    "input": {"name": IO_NAMES[0], "dtype": "float32", "shape": [-1, N_FEATURES]},
    "output": {"name": IO_NAMES[1], "dtype": "float32", "classes": CLASSES},
    "features": features,
    "numeric": {
        "cols": enc["num_cols"],
        "note": "has_*/dhcp_opt_* là 0/1; dhcp_prl_len = số option khác nhau trong dhcp_prl; "
                "mac_is_random = bit 0x02 của octet đầu MAC (MAC hỏng -> 0). Thiếu -> 0.",
    },
    "categorical": {"unknown_index": -1, "categories": categorical},
    "tfidf": tfidf,
    "decision": {
        "order": ["l0_hostname", "l0_vci", "l1_prl", "l1_mdns", "forest"],
        "unknown_label": OS05B["UNK"],
        "l0_hostname": {"lowercase": True, "strip": True, "ignore": [MISSING, "nan", "none", ""],
                        "rules": [{"regex": p, "os": o} for p, o in OS05B["HOST_OS"]]},
        "l0_vci": [{"prefix": "msft ", "os": "Windows", "lowercase": True},
                   {"prefix": "android-dhcp-", "os": "Android", "lowercase": True,
                    "require_mac_random": True}],
        "l1_prl": {"min_jaccard": 0.9, "signatures": signatures,
                   "rule": "Jaccard cao nhất (hoà thì gộp tập OS); trả lời khi >= min_jaccard và đúng 1 OS; "
                           "bỏ qua nếu OS là Embedded mà MAC ngẫu nhiên."},
        "l1_mdns": {"require_mac_random": True, "contains_any": ["_companion link", "_rdlink"], "os": "iOS"},
        "forest": {"threshold": float(bundle["threshold"]),
                   "rule": "argmax proba; trả lời khi proba >= threshold, trừ khi lớp khác Embedded và "
                           "cửa sổ không có DHCP lẫn mDNS (has_dhcp == 0 và has_mdns == 0)."},
    },
}


def encode_from_contract(row, k=contract):
    """Encoder tham chiếu chỉ dùng contract — hình mẫu cho code phía thiết bị."""
    x = np.zeros(len(k["features"]), dtype=np.float32)
    for i, c in enumerate(k["numeric"]["cols"]):
        x[i] = float(row.get(c, 0) or 0)
    base = len(k["numeric"]["cols"])
    for j, (c, cats) in enumerate(k["categorical"]["categories"].items()):
        v = str(row.get(c, k["missing"]))
        x[base + j] = cats.index(v) if v in cats else k["categorical"]["unknown_index"]
    for c, spec in k["tfidf"].items():
        text = str(row.get(c, k["missing"]))
        text = text.lower() if spec["lowercase"] else text
        pos = {t: i for i, t in enumerate(spec["vocabulary"])}
        block = np.zeros(len(pos))
        for tok in re.findall(spec["token_pattern"], text):
            if tok in pos:
                block[pos[tok]] += 1.0
        block *= np.asarray(spec["idf"])
        norm = np.linalg.norm(block)
        if spec["norm"] == "l2" and norm > 0:
            block /= norm
        x[spec["offset"]:spec["offset"] + len(pos)] = block
    return x


worst = 0.0
for name, frame in frames.items():
    X_ct = apply_encoder(frame[bundle["cols"]], enc)[0]
    X_k = np.stack([encode_from_contract(r) for r in frame.to_dict("records")])
    diff = float(np.abs(X_ct - X_k).max())
    worst = max(worst, diff)
    print(f"{name:22s} {len(frame):5d} dòng  max |Δx| = {diff:.2e}")
assert worst < 1e-6

sessions_verified       8189 dòng  max |Δx| = 0.00e+00
capture                   62 dòng  max |Δx| = 0.00e+00
test_pcap_collection       3 dòng  max |Δx| = 0.00e+00


## 6. Nhúng contract vào `.tflite`

Thêm một buffer chứa JSON và một mục `metadata` tên `sdc.contract` trỏ tới nó — graph không đổi. Kiểm lại
từ **file đã nhúng**: đọc contract bằng flatbuffer thuần (đúng cách phía thiết bị làm), so từng byte, chạy lại
encoder tham chiếu bằng contract đọc ra, và output của graph phải giống hệt bản chưa nhúng.

In [8]:
import hashlib

CONTRACT_KEY = "sdc.contract"


def embed_metadata(model_bytes, name, payload):
    model = flatbuffer_utils.read_model_from_bytearray(bytearray(model_bytes))
    names = [m.name.decode() if isinstance(m.name, bytes) else m.name for m in model.metadata or []]
    assert name not in names, f"metadata {name!r} đã có"
    buf = tfl_schema.BufferT()
    buf.data = np.frombuffer(payload, dtype=np.uint8)
    model.buffers.append(buf)
    entry = tfl_schema.MetadataT()
    entry.name, entry.buffer = name, len(model.buffers) - 1
    model.metadata = (model.metadata or []) + [entry]
    return bytes(flatbuffer_utils.convert_object_to_bytearray(model))


def read_metadata(model_bytes, name):
    """Chỉ dùng schema flatbuffer — không cần TF runtime, như code phía thiết bị."""
    model = tfl_schema.Model.GetRootAsModel(model_bytes, 0)
    for i in range(model.MetadataLength()):
        entry = model.Metadata(i)
        if entry.Name().decode() == name:
            return model.Buffers(entry.Buffer()).DataAsNumpy().tobytes()
    raise KeyError(name)


contract_bytes = json.dumps(contract, indent=1, ensure_ascii=False).encode("utf-8")
contract_sha256 = hashlib.sha256(contract_bytes).hexdigest()
final_bytes = embed_metadata(tflite_bytes, CONTRACT_KEY, contract_bytes)

embedded = read_metadata(final_bytes, CONTRACT_KEY)
assert embedded == contract_bytes
contract_back = json.loads(embedded)
for name, frame in frames.items():
    X_ct = apply_encoder(frame[bundle["cols"]], enc)[0]
    X_k = np.stack([encode_from_contract(r, contract_back) for r in frame.to_dict("records")])
    assert np.array_equal(X_ct, X_k), name
X_chk = random_encoded(2_000, seed=1)
assert np.array_equal(tflite_proba(X_chk), tflite_proba(X_chk, final_bytes))
_it = Interpreter(model_content=final_bytes)
assert (_it.get_input_details()[0]["name"], _it.get_output_details()[0]["name"]) == IO_NAMES
print(f"contract {len(contract_bytes) / 1024:.1f} KiB nhúng vào metadata {CONTRACT_KEY!r}: "
      f"{len(tflite_bytes) / 1024:.0f} -> {len(final_bytes) / 1024:.0f} KiB, sha256 {contract_sha256[:16]}…")

contract 38.3 KiB nhúng vào metadata 'sdc.contract': 485 -> 523 KiB, sha256 9951d5a229497d18…


## 7. Lưu

In [9]:
out_dir = run_dir / "tflite"
out_dir.mkdir(exist_ok=True)
(out_dir / "sdc_os_forest.tflite").write_bytes(final_bytes)
(out_dir / "sdc_os_contract.json").write_bytes(contract_bytes)   # bản sao để đọc, cùng sha256
(out_dir / "meta.json").write_text(json.dumps({
    "format": "sdc-os-tflite-v1",
    "source_run": run_dir.name,
    "created": contract["created"],
    "tensorflow": tf.__version__,
    "parity_runtime": RUNTIME,
    "tflite_bytes": len(final_bytes),
    "contract_metadata_key": CONTRACT_KEY,
    "contract_sha256": contract_sha256,
    "ops": ops,
    "trees": int(len(flat["roots"])), "nodes": int(len(flat["feature"])), "depth": int(flat["depth"]),
    "parity": parity.reset_index().to_dict("records"),
    "encoder_parity_max_abs": worst,
}, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
for p in sorted(out_dir.iterdir()):
    print(f"{p.stat().st_size / 1024:8.1f} KiB  {p}")

     1.3 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite\meta.json
    38.3 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite\sdc_os_contract.json
   523.4 KiB  D:\01.AI_Security\02.SDC\Models\20261002_163153_os\tflite\sdc_os_forest.tflite
